# Nepali Cultural Dress — Image Captioning
### Tiny ViT Encoder + Transformer Decoder

Trains an encoder–decoder captioning model **entirely from scratch** (no pretrained
weights) on the Nepali dress dataset.

**Architecture**
```
Image → Patch Embedding → Positional Encoding
    → Transformer Encoder (4 Layers)
    → Encoder Memory
    → Cross Attention
    → Transformer Decoder (4 Layers)
    → Linear + Softmax → Caption
```

**How to use on Kaggle**
1. Create a Kaggle Dataset containing `captions.csv` and the `nepali_dresses_augmented/`
   folder (keep the folder structure so the relative `image_path` values in the CSV resolve).
2. Add that dataset to the notebook (right panel → *Add Input*).
3. In **Section 1 (Config)** set `CFG.DATA_DIR` to your dataset's mount path
   (e.g. `/kaggle/input/nepali-dress-captioning`).
4. Turn on GPU (Settings → Accelerator → GPU) and *Run All*.

> **Everything you'd want to tune lives in the single Config cell below.**
> You should not need to edit any other cell for normal experiments.


## 1. Configuration  ⚙️ *(edit only here)*

In [ ]:
import os

class CFG:
    # ---------------- Reproducibility ----------------
    SEED = 42

    # ---------------- Paths (Kaggle) ----------------
    # DATA_DIR must contain captions.csv AND the nepali_dresses_augmented/ folder.
    DATA_DIR   = "."                     # <-- CHANGE THIS: dir with nepali_dresses_augmented/ and captions.csv
    CSV_DIR    = "."                     # dir containing captions.csv
    CSV_NAME   = "captions.csv"
    CAPTION_COL = "caption_en"        # "caption_en" or "caption_ne"
    OUTPUT_DIR = "tinyvit_output"      # writable dir for checkpoints/logs

    # ---------------- Data & split ----------------
    VAL_FRACTION    = 0.10
    TEST_FRACTION   = 0.10
    SPLIT_BY_SOURCE = True
    IMAGE_SIZE      = 224
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD  = [0.229, 0.224, 0.225]
    USE_TRAIN_AUG = True

    # ---------------- Vocabulary ----------------
    MIN_WORD_FREQ   = 2
    MAX_CAPTION_LEN = 22

    # ---------------- Tiny ViT Encoder ----------------
    PATCH_SIZE      = 16        # 224/16 = 14x14 = 196 patches
    HIDDEN_DIM      = 256       # embedding / transformer dimension
    ENC_NHEAD       = 8         # attention heads (must divide HIDDEN_DIM)
    ENC_LAYERS      = 4         # transformer encoder layers
    ENC_DIM_FEED    = 512       # FFN hidden size
    ENC_DROPOUT     = 0.1

    # ---------------- Transformer Decoder ----------------
    DEC_NHEAD       = 8
    DEC_LAYERS      = 4
    DEC_DIM_FEED    = 512
    DEC_DROPOUT     = 0.1

    # ---------------- Training ----------------
    EPOCHS      = 100
    BATCH_SIZE  = 32
    ENCODER_LR  = 1e-4
    DECODER_LR  = 4e-4
    GRAD_CLIP   = 5.0
    NUM_WORKERS = 0

    # ---------------- Early stopping & checkpoints ----------------
    EARLY_STOPPING = True
    PATIENCE       = 8
    MONITOR        = "bleu4"
    CKPT_BEST      = "best_model.pth"
    CKPT_LAST      = "last_model.pth"

    # ---------------- Resume ----------------
    RESUME      = False
    RESUME_PATH = "tinyvit_output/last_model.pth"

    # ---------------- Inference / evaluation ----------------
    BEAM_SIZE = 3

print({k: v for k, v in vars(CFG).items() if not k.startswith("__")})

## 2. Imports & setup

In [ ]:
import os, re, json, random, time
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print("Torch:", torch.__version__, "| Device:", DEVICE)

## 3. Load captions & leakage-safe split

In [ ]:
csv_path = os.path.join(CFG.CSV_DIR, CFG.CSV_NAME)
df = pd.read_csv(csv_path)
assert CFG.CAPTION_COL in df.columns, f"{CFG.CAPTION_COL} not in {list(df.columns)}"
print("total rows:", len(df))

def parse_source(path):
    cls = os.path.basename(os.path.dirname(path))
    fn  = os.path.basename(path)
    m = re.match(r"(\d+)(?:_aug\d+)?\.jpg", fn, re.I)
    stem = m.group(1) if m else os.path.splitext(fn)[0]
    return cls, f"{cls}/{stem}"

df["cls"], df["source"] = zip(*df["image_path"].map(parse_source))

groups = df["source"] if CFG.SPLIT_BY_SOURCE else pd.Series(df.index.astype(str), index=df.index)
uniq = sorted(groups.unique())
rng = random.Random(CFG.SEED); rng.shuffle(uniq)
n = len(uniq)
n_test = int(n * CFG.TEST_FRACTION)
n_val  = int(n * CFG.VAL_FRACTION)
test_g = set(uniq[:n_test])
val_g  = set(uniq[n_test:n_test + n_val])

def which(g):
    if g in test_g: return "test"
    if g in val_g:  return "val"
    return "train"

df["split"] = groups.map(which)
print(df["split"].value_counts())
print("\nper-class / split:\n", df.groupby(["split", "cls"]).size().unstack(fill_value=0))

## 4. Vocabulary (built from the **train** split only)

In [ ]:
PAD, START, END, UNK = "<pad>", "<start>", "<end>", "<unk>"

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9ऀ-ॿ]+", " ", text)
    return text.split()

class Vocab:
    def __init__(self, freqs, min_freq):
        self.itos = [PAD, START, END, UNK]
        for w, c in sorted(freqs.items(), key=lambda x: (-x[1], x[0])):
            if c >= min_freq and w not in (PAD, START, END, UNK):
                self.itos.append(w)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len):
        toks = [START] + tokenize(text)[: max_len - 2] + [END]
        length = len(toks)
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in toks]
        ids += [self.stoi[PAD]] * (max_len - length)
        return ids, length

freqs = Counter()
for c in df.loc[df.split == "train", CFG.CAPTION_COL]:
    freqs.update(tokenize(c))
vocab = Vocab(freqs, CFG.MIN_WORD_FREQ)
pad_idx, start_idx, end_idx = vocab.stoi[PAD], vocab.stoi[START], vocab.stoi[END]
print("vocab size:", len(vocab))
print("sample words:", vocab.itos[4:24])

## 5. Transforms, Dataset & DataLoaders

In [ ]:
_aug = [T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.1)] if CFG.USE_TRAIN_AUG else []
train_tf = T.Compose([T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)), *_aug,
                      T.ToTensor(), T.Normalize(CFG.NORM_MEAN, CFG.NORM_STD)])
eval_tf  = T.Compose([T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
                      T.ToTensor(), T.Normalize(CFG.NORM_MEAN, CFG.NORM_STD)])

class CaptionDataset(Dataset):
    def __init__(self, frame, vocab, transform):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(os.path.join(CFG.DATA_DIR, row["image_path"])).convert("RGB")
        img = self.transform(img)
        ids, length = self.vocab.encode(row[CFG.CAPTION_COL], CFG.MAX_CAPTION_LEN)
        return img, torch.tensor(ids), torch.tensor(length)

def make_loader(split, transform, shuffle):
    ds = CaptionDataset(df[df.split == split], vocab, transform)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=shuffle,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=shuffle)

train_loader = make_loader("train", train_tf, True)
val_loader   = make_loader("val",   eval_tf, False)
test_loader  = make_loader("test",  eval_tf, False)
print("batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

## 6. Model

### 6a. Tiny ViT Encoder

Splits the image into 16×16 patches, embeds each patch, adds positional
embeddings, and applies 4 Transformer Encoder layers.  The output serves
as the memory for the decoder's cross-attention.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=256):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)                           # [B, D, H/p, W/p]
        x = x.flatten(2).transpose(1, 2)           # [B, num_patches, D]
        return x

class TinyViTEncoder(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=256,
                 depth=4, nhead=8, dim_feedforward=512, dropout=0.1,
                 image_size=224):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim)
        num_patches = (image_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            embed_dim, nhead, dim_feedforward, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, depth,
                                             enable_nested_tensor=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)                     # [B, num_patches, D]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)       # [B, num_patches+1, D]
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = self.encoder(x)                         # [B, num_patches+1, D]
        return x

### 6b. Transformer Decoder with Cross-Attention

Takes the encoder memory and applies 4 Transformer Decoder layers.
Each layer has:
1. Masked self-attention over the target sequence (causal)
2. Cross-attention over the encoder memory
3. Feed-forward network

In [ ]:
class CaptionTransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, depth=4, nhead=8,
                 dim_feedforward=512, dropout=0.1, max_len=22):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            embed_dim, nhead, dim_feedforward, dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, depth)

        self.fc = nn.Linear(embed_dim, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for p in [self.fc.weight, self.embedding.weight]:
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        self.fc.bias.data.fill_(0)

    def forward(self, tgt, memory, tgt_mask=None, tgt_padding_mask=None):
        tgt = self.embedding(tgt)                         # [B, T, D]
        tgt = tgt + self.pos_embed[:, :tgt.size(1), :]
        tgt = self.pos_drop(tgt)

        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                tgt.size(1), device=tgt.device)

        out = self.decoder(tgt, memory,
                           tgt_mask=tgt_mask,
                           memory_key_padding_mask=None,
                           tgt_key_padding_mask=tgt_padding_mask)
        return self.fc(out)                                # [B, T, vocab_size]

## 7. Training utilities

In [ ]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n; self.avg = self.sum / self.count

def clip_gradient(optimizer, grad_clip):
    for group in optimizer.param_groups:
        for p in group["params"]:
            if p.grad is not None:
                p.grad.data.clamp_(-grad_clip, grad_clip)

def save_checkpoint(path, epoch, best_score):
    torch.save({
        "epoch": epoch,
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "enc_opt": enc_opt.state_dict(),
        "dec_opt": dec_opt.state_dict(),
        "best_score": best_score,
        "vocab_itos": vocab.itos,
        "config": {k: v for k, v in vars(CFG).items() if not k.startswith("__")},
    }, path)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx).to(DEVICE)

## 8. Train / validate functions

The decoder is trained with **teacher forcing**: the full target caption is fed
as input (shifted right), and a causal mask prevents the model from peeking at
future tokens.

In [ ]:
def run_epoch(loader, train):
    encoder.train(train); decoder.train(train)
    losses = AverageMeter()
    references, hypotheses = [], []
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc=f"{'train' if train else 'val':5s}", leave=False)
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for imgs, caps, lengths in pbar:
            imgs, caps, lengths = imgs.to(DEVICE), caps.to(DEVICE), lengths.to(DEVICE)
            B, T = caps.shape

            memory = encoder(imgs)                           # [B, num_patches+1, D]

            # Teacher forcing: input = tokens[:-1], target = tokens[1:]
            tgt_input  = caps[:, :-1]                        # [B, T-1]
            tgt_target = caps[:, 1:]                         # [B, T-1]

            # Padding mask for target (positions that are pad)
            tgt_padding_mask = (tgt_input == pad_idx)        # [B, T-1]

            logits = decoder(tgt_input, memory,
                             tgt_padding_mask=tgt_padding_mask)  # [B, T-1, V]

            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             tgt_target.reshape(-1))

            if train:
                enc_opt.zero_grad(); dec_opt.zero_grad()
                loss.backward()
                if CFG.GRAD_CLIP:
                    clip_gradient(enc_opt, CFG.GRAD_CLIP)
                    clip_gradient(dec_opt, CFG.GRAD_CLIP)
                enc_opt.step(); dec_opt.step()

            losses.update(loss.item(), B)
            pbar.set_postfix(loss=losses.avg)

            if not train:
                words = logits.argmax(dim=-1)
                for j in range(B):
                    ref = caps[j].tolist()
                    ref = [w for w in ref if w not in (pad_idx, start_idx, end_idx)]
                    hyp = words[j].tolist()
                    hyp = [w for w in hyp if w not in (pad_idx, start_idx, end_idx)]
                    references.append([ref]); hypotheses.append(hyp)

    if train:
        return losses.avg, None
    smooth = SmoothingFunction().method1
    bleu4 = corpus_bleu(references, hypotheses, smoothing_function=smooth)
    return losses.avg, bleu4

## 9. Build models, optimizers (+ optional resume)

In [ ]:
encoder = TinyViTEncoder(
    in_channels=3, patch_size=CFG.PATCH_SIZE, embed_dim=CFG.HIDDEN_DIM,
    depth=CFG.ENC_LAYERS, nhead=CFG.ENC_NHEAD,
    dim_feedforward=CFG.ENC_DIM_FEED, dropout=CFG.ENC_DROPOUT,
    image_size=CFG.IMAGE_SIZE).to(DEVICE)

decoder = CaptionTransformerDecoder(
    len(vocab), embed_dim=CFG.HIDDEN_DIM, depth=CFG.DEC_LAYERS,
    nhead=CFG.DEC_NHEAD, dim_feedforward=CFG.DEC_DIM_FEED,
    dropout=CFG.DEC_DROPOUT, max_len=CFG.MAX_CAPTION_LEN).to(DEVICE)

enc_opt = torch.optim.Adam(encoder.parameters(), lr=CFG.ENCODER_LR)
dec_opt = torch.optim.Adam(decoder.parameters(), lr=CFG.DECODER_LR)

n_params = sum(p.numel() for p in encoder.parameters()) + \
           sum(p.numel() for p in decoder.parameters())
print(f"trainable params: {n_params/1e6:.1f}M")
print(f"  encoder: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")
print(f"  decoder: {sum(p.numel() for p in decoder.parameters())/1e6:.1f}M")

start_epoch = 0
best_score  = -1e9 if CFG.MONITOR == "bleu4" else 1e9
epochs_no_improve = 0
if CFG.RESUME and os.path.exists(CFG.RESUME_PATH):
    ck = torch.load(CFG.RESUME_PATH, map_location=DEVICE)
    encoder.load_state_dict(ck["encoder"]); decoder.load_state_dict(ck["decoder"])
    enc_opt.load_state_dict(ck["enc_opt"]); dec_opt.load_state_dict(ck["dec_opt"])
    start_epoch = ck["epoch"] + 1; best_score = ck["best_score"]
    print(f"Resumed from epoch {start_epoch} (best={best_score:.4f})")

## 10. Training loop — early stopping + checkpoints

In [ ]:
def is_better(new, best):
    return new > best if CFG.MONITOR == "bleu4" else new < best

from tqdm.auto import tqdm
history = []
epoch_bar = tqdm(range(start_epoch, CFG.EPOCHS), desc="Epochs")
for epoch in epoch_bar:
    t0 = time.time()
    train_loss, _        = run_epoch(train_loader, train=True)
    val_loss, val_bleu4  = run_epoch(val_loader,   train=False)
    score = val_bleu4 if CFG.MONITOR == "bleu4" else val_loss
    dt = time.time() - t0
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_bleu4": val_bleu4})
    epoch_bar.set_postfix(train=train_loss, val=val_loss, bleu=val_bleu4)
    print(f"Epoch {epoch+1:02d}/{CFG.EPOCHS} | train {train_loss:.3f} | "
          f"val {val_loss:.3f} | BLEU-4 {val_bleu4:.4f} | {dt:.0f}s")

    save_checkpoint(os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_LAST), epoch, best_score)
    if is_better(score, best_score):
        best_score = score; epochs_no_improve = 0
        save_checkpoint(os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_BEST), epoch, best_score)
        print(f"   new best ({CFG.MONITOR} = {best_score:.4f}) -> saved best_model.pth")
    else:
        epochs_no_improve += 1
        print(f"   no improvement ({epochs_no_improve}/{CFG.PATIENCE})")
        if CFG.EARLY_STOPPING and epochs_no_improve >= CFG.PATIENCE:
            print("Early stopping triggered."); break

with open(os.path.join(CFG.OUTPUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("done. best score:", best_score)

## 11. Training curves

In [ ]:
import matplotlib.pyplot as plt
if 'history' in dir() and history:
    ep = [h["epoch"] + 1 for h in history]
    plt.figure(figsize=(10, 4))
    plt.plot(ep, [h["train_loss"] for h in history], "o-", label="train", color="#3498db", linewidth=1.5, markersize=3)
    plt.plot(ep, [h["val_loss"] for h in history], "s-", label="val", color="#e74c3c", linewidth=1.5, markersize=3)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
    plt.title("Training Curves"); plt.tight_layout(); plt.show()

## 12. Inference — greedy caption generation

Loads `best_model.pth` and generates a caption with greedy decoding.
The decoder starts with `<start>`, then iteratively predicts the next token.

In [ ]:
import os
def load_best():
    p = os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_BEST)
    if os.path.exists(p):
        ck = torch.load(p, map_location=DEVICE)
        encoder.load_state_dict(ck["encoder"]); decoder.load_state_dict(ck["decoder"])
        print("loaded", p)
load_best()

@torch.no_grad()
def caption_image(image_path, max_len=CFG.MAX_CAPTION_LEN):
    encoder.eval(); decoder.eval()
    img = eval_tf(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)

    memory = encoder(img)                                   # [1, num_patches+1, D]

    seq = torch.full((1, 1), start_idx, dtype=torch.long, device=DEVICE)
    for _ in range(1, max_len):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            seq.size(1), device=DEVICE)
        logits = decoder(seq, memory, tgt_mask=tgt_mask)    # [1, T, V]
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        seq = torch.cat([seq, next_token], dim=1)
        if next_token.item() == end_idx:
            break

    words = [vocab.itos[w] for w in seq[0].tolist()
             if w not in (pad_idx, start_idx, end_idx)]
    return " ".join(words)

## 13. Show predictions on random test images

In [ ]:
sample = df[df.split == "test"].sample(min(4, (df.split == "test").sum()), random_state=CFG.SEED)
plt.figure(figsize=(12, 10))
for i, (_, row) in enumerate(sample.iterrows()):
    p = os.path.join(CFG.DATA_DIR, row["image_path"])
    pred = caption_image(p)
    ax = plt.subplot(2, 2, i + 1)
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    ax.set_title(f"pred: {pred}\ngt: {row[CFG.CAPTION_COL][:70]}", fontsize=9)
plt.tight_layout(); plt.show()

## 14. Final BLEU on the test set

In [ ]:
@torch.no_grad()
def evaluate_bleu(split="test", max_len=CFG.MAX_CAPTION_LEN, limit=None):
    rows = df[df.split == split]
    if limit:
        rows = rows.sample(min(limit, len(rows)), random_state=CFG.SEED)
    refs, hyps = [], []
    for _, row in rows.iterrows():
        gt   = tokenize(row[CFG.CAPTION_COL])
        pred = caption_image(os.path.join(CFG.DATA_DIR, row["image_path"]),
                             max_len).split()
        refs.append([gt]); hyps.append(pred)
    smooth = SmoothingFunction().method1
    b1 = corpus_bleu(refs, hyps, weights=(1, 0, 0, 0),          smoothing_function=smooth)
    b2 = corpus_bleu(refs, hyps, weights=(0.5, 0.5, 0, 0),      smoothing_function=smooth)
    b3 = corpus_bleu(refs, hyps, weights=(1/3, 1/3, 1/3, 0),    smoothing_function=smooth)
    b4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    print(f"{split}: BLEU-1 {b1:.4f} | BLEU-2 {b2:.4f} | BLEU-3 {b3:.4f} | BLEU-4 {b4:.4f}")
    return b1, b2, b3, b4

evaluate_bleu("test", limit=200)

## Notes

- **From scratch is hard on this data.** The Tiny ViT + Transformer has
  ~{}M params and is trained on ~3.6k images; expect modest BLEU scores.
  That is inherent to the "no pretrained weights" constraint.
- The val BLEU used for early stopping is *teacher-forced* (cheap).
  Section 14 gives the true greedy BLEU.
- Checkpoints + `history.json` are written to `/kaggle/working`.


In [ ]:
# ===== Report Visualizations =====
import matplotlib.pyplot as plt
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from PIL import Image

has_history = "history" in dir() and len(history) > 0

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3)

ax1 = fig.add_subplot(gs[0, :2])
if has_history:
    ep = [h["epoch"] + 1 for h in history]
    ax1.plot(ep, [h["train_loss"] for h in history], "o-", label="Train Loss", color="#e74c3c")
    ax1.plot(ep, [h["val_loss"] for h in history], "s-", label="Val Loss", color="#3498db")
    ax1.set_ylabel("Loss", color="#c0392b")
    ax1.tick_params(axis="y", labelcolor="#c0392b")
    ax1.legend(loc="upper left")
    ax1.set_title("Training & Validation Loss", fontsize=13, fontweight="bold")

    ax1b = ax1.twinx()
    ax1b.plot(ep, [h["val_bleu4"] for h in history], "d-", label="Val BLEU-4", color="#2ecc71", linewidth=2)
    ax1b.set_ylabel("BLEU-4", color="#27ae60")
    ax1b.tick_params(axis="y", labelcolor="#27ae60")
    ax1b.legend(loc="upper right")
    ax1.set_xlabel("Epoch")
    ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 2])
try:
    b_scores = evaluate_bleu("test", limit=200)
    b_labels = ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]
    colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]
    bars = ax2.barh(b_labels, b_scores, color=colors, edgecolor="white", height=0.6)
    for bar, score in zip(bars, b_scores):
        ax2.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{score:.4f}", va="center", fontsize=10)
    ax2.set_xlim(0, max(b_scores) + 0.05)
    ax2.set_title("Test Set BLEU Scores", fontsize=13, fontweight="bold")
    ax2.grid(True, axis="x", alpha=0.3)
except Exception as e:
    ax2.text(0.5, 0.5, f"BLEU eval failed:\n{e}", ha="center", va="center", transform=ax2.transAxes, fontsize=10)

ax3 = fig.add_subplot(gs[1, :])
try:
    classes = sorted(df[df.split == "test"]["cls"].unique())
    cls_scores = []
    for cls in classes:
        rows = df[(df.split == "test") & (df["cls"] == cls)]
        refs, hyps = [], []
        for _, row in rows.iterrows():
            gt = tokenize(row[CFG.CAPTION_COL])
            pred = caption_image(os.path.join(CFG.DATA_DIR, row["image_path"])).split()
            refs.append([gt]); hyps.append(pred)
        smooth = SmoothingFunction().method1
        b4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        cls_scores.append(b4)

    cls_labels = [c.replace("_", " ").title() for c in classes]
    bar_colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))
    bars = ax3.bar(cls_labels, cls_scores, color=bar_colors, edgecolor="gray", linewidth=0.5)
    for bar, score in zip(bars, cls_scores):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{score:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax3.set_title("Per-Class BLEU-4 on Test Set", fontsize=13, fontweight="bold")
    ax3.set_ylabel("BLEU-4")
    ax3.set_ylim(0, max(cls_scores) + 0.05)
    ax3.grid(True, axis="y", alpha=0.3)
    plt.setp(ax3.get_xticklabels(), rotation=30, ha="right", fontsize=9)
except Exception as e:
    ax3.text(0.5, 0.5, f"Per-class BLEU failed:\n{e}", ha="center", va="center", transform=ax3.transAxes, fontsize=10)

ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
ax6 = fig.add_subplot(gs[2, 2])
try:
    test_samples = df[df.split == "test"].sample(3, random_state=42)
    for ax_i, (_, row) in zip([ax4, ax5, ax6], test_samples.iterrows()):
        img_path = os.path.join(CFG.DATA_DIR, row["image_path"])
        img = Image.open(img_path).convert("RGB")
        pred = caption_image(img_path)
        gt = row[CFG.CAPTION_COL]
        ax_i.imshow(img)
        ax_i.axis("off")
        ax_i.set_title(f"GT: {gt[:50]}\nPred: {pred[:50]}", fontsize=8, pad=10)
except Exception as e:
    for ax_i in [ax4, ax5, ax6]:
        ax_i.text(0.5, 0.5, f"Samples failed:\n{e}", ha="center", va="center", transform=ax_i.transAxes, fontsize=10)
        ax_i.axis("off")

plt.suptitle("Nepali Cultural Dress Captioning — Tiny ViT + Transformer Decoder",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
if has_history:
    best_idx = np.argmax([h["val_bleu4"] for h in history])
    best_epoch = history[best_idx]["epoch"] + 1
    print(f"Best epoch:           {best_epoch}")
    print(f"Best val BLEU-4:      {history[best_idx]['val_bleu4']:.4f}")
    print(f"Final train loss:     {history[-1]['train_loss']:.4f}")
    print(f"Final val loss:       {history[-1]['val_loss']:.4f}")
    print(f"Total epochs trained: {len(history)}")
try:
    b1, b2, b3, b4 = b_scores
    print(f"Test BLEU-1:           {b1:.4f}")
    print(f"Test BLEU-2:           {b2:.4f}")
    print(f"Test BLEU-3:           {b3:.4f}")
    print(f"Test BLEU-4:           {b4:.4f}")
except:
    pass
print(f"Vocabulary size:      {len(vocab)}")
print(f"Dataset size:         {len(df)} ({len(df[df.split=='train'])} train / {len(df[df.split=='val'])} val / {len(df[df.split=='test'])} test)")
print(f"Classes:              {len(df['cls'].unique())}")
total_params = sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters())
print(f"Model parameters:     {total_params/1e6:.1f}M")
print("=" * 60)
